# 09 Data Augmentation Impact Analysis
This notebook compares a **Baseline Model** (no augmentation) against an **Augmented Model** (heavy transformations) to evaluate robustness and generalization.

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import mlflow
from sklearn.metrics import f1_score, recall_score
import matplotlib.pyplot as plt
import numpy as np

from pneumonia_classifier.ml.model.arch import Net

## Configuration

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = r'j:\Users\ayush\Desktop\code\pneumonia_classifier\artifacts\02_12_2025_08_52_04\data_ingestion\data\data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
INPUT_SIZE = (224, 224)
BATCH_SIZE = 8
EPOCHS = 5

mlflow.set_experiment("augmentation_impact_analysis")

<Experiment: artifact_location='file:///j:/Users/ayush/Desktop/code/pneumonia_classifier/notebooks/mlruns/2', creation_time=1772278540841, experiment_id='2', last_update_time=1772278540841, lifecycle_stage='active', name='augmentation_impact_analysis', tags={}, workspace='default'>

## Define Transforms

In [3]:
baseline_transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

augmented_transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Training Loop

In [4]:
def run_experiment(name, transform):
    print(f"Starting Experiment: {name}")
    with mlflow.start_run(run_name=name):
        train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
        test_dataset = datasets.ImageFolder(TEST_DIR, transform=baseline_transform) # Always test on clean data
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        model = Net().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=0.0004)
        
        for epoch in range(1, EPOCHS + 1):
            model.train()
            train_loss = 0
            for data, target in train_loader:
                data, target = data.to(DEVICE), target.to(DEVICE)
                optimizer.zero_grad()
                output = model(data)
                loss = F.nll_loss(output, target)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for data, target in test_loader:
                    data, target = data.to(DEVICE), target.to(DEVICE)
                    output = model(data)
                    pred = output.argmax(dim=1).cpu().numpy()
                    all_preds.extend(pred)
                    all_targets.extend(target.cpu().numpy())
            
            f1 = f1_score(all_targets, all_preds)
            mlflow.log_metric("f1_score", f1, step=epoch)
            print(f"Epoch {epoch}: F1 {f1:.4f}")
        
        mlflow.pytorch.log_model(model, "model")
        print(f"Experiment {name} complete.\n")

## Run Comparisons

In [5]:
run_experiment("Baseline_No_Augmentation", baseline_transform)
run_experiment("Augmented_Heavy", augmented_transform)

Starting Experiment: Baseline_No_Augmentation
Epoch 1: F1 0.6897
Epoch 2: F1 0.8788
Epoch 3: F1 0.8462
Epoch 4: F1 0.9492


2026/02/28 17:38:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 5: F1 0.9655


2026/02/28 17:38:26 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Experiment Baseline_No_Augmentation complete.

Starting Experiment: Augmented_Heavy
Epoch 1: F1 0.6897
Epoch 2: F1 0.7586
Epoch 3: F1 0.9333
Epoch 4: F1 0.9153


2026/02/28 17:42:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 5: F1 0.8889


2026/02/28 17:42:14 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Experiment Augmented_Heavy complete.



## Conclusion
Open the MLflow UI to compare the convergence speed and final F1-Score. Generally, augmentation improves robustness against real-world variations.